In [1]:
!mamba install pandas
!mamba install scikit-learn

mambajs 0.21.1

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 3.27689999999851 seconds
  Name           Version  Build                Channel
--------------------------------------------------------------------
+ pandas         3.0.3    np23py313h1e705a5_0  emscripten-forge-4x
+ python-tzdata  2026.2   pyhd8ed1ab_0         conda-forge
- pip            26.1.2   pyh145f28c_0         conda-forge
mambajs 0.21.1

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, scikit-learn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 1.2205 seconds
  Name                Version    Build                Channel
---------------------------------------------------------------------------
+ brotli-python       1.2.0      py313ha26e73d_2      emscripten-forge-4x
+ certifi             2026.5.20  pyhd8ed1ab_0    

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

import time
import pickle

# 1. DATA LOADING & PREPARATION
print("Loading data...")
# Read the CSV files and assign labels (0 for Fake, 1 for True)
fake_df = pd.read_csv('Fake.csv')
fake_df['label'] = 0 

true_df = pd.read_csv('True.csv')
true_df['label'] = 1

# Combine and shuffle the dataset
data = pd.concat([fake_df, true_df], axis=0, ignore_index=True)
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Extract features and labels
X = data['text']
y = data['label']

# Split into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

# 2. BUILDING THE 5-MODEL ENSEMBLE
print("Building the 5-Model Ensemble...")

# Define the individual "voter" models
model_1 = LogisticRegression(max_iter=1000)
model_2 = RandomForestClassifier(n_estimators=100, n_jobs=-1, verbose=2)
base_sgd = SGDClassifier(loss='hinge', max_iter=1000, verbose=1, random_state=42)
model_3 = CalibratedClassifierCV(base_sgd)
model_4 = MultinomialNB()
model_5 = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,verbose=1)

# Combine them into a Soft Voting Classifier
ensemble_classifier = VotingClassifier(
    estimators=[
        ('lr', model_1), 
        ('rf', model_2), 
        ('svc', model_3),
        ('nb', model_4),
        ('gb', model_5)
    ],
    voting='soft',
    verbose=True
)

# Pipeline linking TF-IDF and the Ensemble
clf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_df=0.7)),
    ('ensemble', ensemble_classifier)
])

# 3. TRAINING AND EVALUATING
print("Training the ensemble... Grab a coffee, this will take a while!")

start_time = time.time() # Start the stopwatch

print("Training the ensemble (This will take a few minutes due to multiple models)...")
clf_pipeline.fit(X_train, y_train)

end_time = time.time() # Stop the stopwatch
total_time = (end_time - start_time) / 60 # Convert seconds to minutes

print(f"Training Complete! Total time: {total_time:.2f} minutes.")

print("Evaluating the model...")
y_pred = clf_pipeline.predict(X_test)

# Print the performance metrics
print(classification_report(y_test, y_pred))

# 4. SAVING PICKLE FILE FOR THE WEB APP
print("Saving the pipeline...")
with open('fake_news_ensemble_model.pkl', 'wb') as file:
    pickle.dump(clf_pipeline, file)

print("Success! Model saved as 'fake_news_ensemble_model.pkl'.")

Loading data...
Building the 5-Model Ensemble...
Training the ensemble... Grab a coffee, this will take a while!
Training the ensemble (This will take a few minutes due to multiple models)...
[Voting] ....................... (1 of 5) Processing lr, total=   0.9s
building tree 1 of 100
building tree 2 of 100
building tree 3 of 100
building tree 4 of 100
building tree 5 of 100
building tree 6 of 100
building tree 7 of 100
building tree 8 of 100
building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
bu

[Parallel(n_jobs=-1)]: Done  40 tasks      | elapsed:   52.4s


building tree 41 of 100
building tree 42 of 100
building tree 43 of 100
building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
building tree 50 of 100
building tree 51 of 100
building tree 52 of 100
building tree 53 of 100
building tree 54 of 100
building tree 55 of 100
building tree 56 of 100
building tree 57 of 100
building tree 58 of 100
building tree 59 of 100
building tree 60 of 100
building tree 61 of 100
building tree 62 of 100
building tree 63 of 100
building tree 64 of 100
building tree 65 of 100
building tree 66 of 100
building tree 67 of 100
building tree 68 of 100
building tree 69 of 100
building tree 70 of 100
building tree 71 of 100
building tree 72 of 100
building tree 73 of 100
building tree 74 of 100
building tree 75 of 100
building tree 76 of 100
building tree 77 of 100
building tree 78 of 100
building tree 79 of 100
building tree 80 of 100
building tree 81 of 100
building tree 82

[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  2.1min finished


[Voting] ....................... (2 of 5) Processing rf, total= 2.1min
-- Epoch 1
Norm: 8.94, NNZs: 47956, Bias: -1.029468, T: 28734, Avg. loss: 0.086704, Objective: 0.100798
Total training time: 0.05 seconds.
-- Epoch 2
Norm: 1.98, NNZs: 53427, Bias: -0.977655, T: 57468, Avg. loss: 0.055780, Objective: 0.068059
Total training time: 0.09 seconds.
-- Epoch 3
Norm: 8.50, NNZs: 54683, Bias: -1.003779, T: 86202, Avg. loss: 0.050859, Objective: 0.062781
Total training time: 0.15 seconds.
-- Epoch 4
Norm: 20.91, NNZs: 55347, Bias: -0.990039, T: 114936, Avg. loss: 0.048620, Objective: 0.060279
Total training time: 0.20 seconds.
-- Epoch 5
Norm: 19.77, NNZs: 55582, Bias: -0.990698, T: 143670, Avg. loss: 0.047172, Objective: 0.058804
Total training time: 0.24 seconds.
-- Epoch 6
Norm: 19.87, NNZs: 55746, Bias: -0.986037, T: 172404, Avg. loss: 0.046272, Objective: 0.057808
Total training time: 0.27 seconds.
-- Epoch 7
Norm: 7.17, NNZs: 55873, Bias: -0.995224, T: 201138, Avg. loss: 0.045426, Obje

[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:    0.6s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:    1.4s finished


              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4696
           1       0.99      0.99      0.99      4284

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

Saving the pipeline...
Success! Model saved as 'fake_news_ensemble_model.pkl'.
